# Fusion : Méthode statistique + Autoencodeur

In [6]:
import pandas as pd
import numpy as np

print("="*70)
print("ÉTAPE 1 : HARMONISATION TEMPORELLE")
print("="*70)

# Charger les 3 sources
df_stat = pd.read_csv('outputs_stats/stat_annotated_timeseries.csv')
df_ae = pd.read_csv('ae_annotated_timeseries.csv')
df_story = pd.read_csv('outputs_stats/shadow_story_fusion.csv')

# Convertir en datetime
df_stat['timestamp'] = pd.to_datetime(df_stat['timestamp'])
df_ae['timestamp'] = pd.to_datetime(df_ae['timestamp'])

# Choisir résolution commune : 15 minutes (recommandé dans le doc)
freq = '15min'

# Ré-échantillonner les deux DataFrames
df_stat_resampled = df_stat.set_index('timestamp').resample(freq).agg({
    'anomaly_detected': 'max',  # Si au moins 1 anomalie dans la fenêtre
    'shadow_score': 'mean',
    'cluster_id': 'first',
    'hour_of_day': 'first'
}).reset_index()

# Pour l'AE, renommer la colonne anomaly_detected en anomaly_ae avant agrégation
df_ae_temp = df_ae.copy()
df_ae_temp['anomaly_ae'] = df_ae_temp['anomaly_detected']
df_ae_temp['error_normalized'] = df_ae_temp['anomaly_score']

df_ae_resampled = df_ae_temp.set_index('timestamp').resample(freq).agg({
    'reconstruction_error': 'mean',
    'anomaly_ae': 'max',
    'error_normalized': 'mean'
}).reset_index()

# Créer DataFrame MAÎTRE avec fusion des deux
df_master = df_stat_resampled.merge(
    df_ae_resampled,
    on='timestamp',
    how='outer',
    suffixes=('_stat', '_ae')
).sort_values('timestamp')

# Remplir les NaN (cas où un seul modèle a des données)
df_master['anomaly_detected'] = df_master['anomaly_detected'].fillna(0).astype(int)
df_master['anomaly_ae'] = df_master['anomaly_ae'].fillna(0).astype(int)
df_master['shadow_score'] = df_master['shadow_score'].fillna(0)
df_master['error_normalized'] = df_master['error_normalized'].fillna(0)

print(f"✅ DataFrame maître créé : {len(df_master)} points à résolution {freq}")
print(f"Colonnes : {df_master.columns.tolist()}")



ÉTAPE 1 : HARMONISATION TEMPORELLE
✅ DataFrame maître créé : 245588 points à résolution 15min
Colonnes : ['timestamp', 'anomaly_detected', 'shadow_score', 'cluster_id', 'hour_of_day', 'reconstruction_error', 'anomaly_ae', 'error_normalized']


In [7]:
print("\n" + "="*70)
print("ÉTAPE 2 : CALCUL DES 8 FEATURES DE FUSION")
print("="*70)

# Jointure avec Shadow Story pour avoir les métadonnées de cluster
df_master = df_master.merge(
    df_story[['cluster_id', 'taux_recurrence', 'intensite_moyenne_pct', 
              'heure_debut', 'heure_fin', 'nb_jours_actifs']],
    on='cluster_id',
    how='left'
)

# --- FEATURES DU STATISTIQUE ---

# Feature 1: Persistance du cluster
df_master['feat_persistance_cluster'] = df_master['taux_recurrence'].fillna(0)

# Feature 2: Proximité heure typique (timestamp dans la plage horaire du cluster?)
def is_in_typical_hours(row):
    if pd.isna(row['heure_debut']) or pd.isna(row['heure_fin']):
        return 0
    hour = row['timestamp'].hour
    h_debut = int(row['heure_debut'].split(':')[0])
    h_fin = int(row['heure_fin'].split(':')[0])
    return 1 if h_debut <= hour <= h_fin else 0

df_master['feat_proximite_heure_typique'] = df_master.apply(is_in_typical_hours, axis=1)

# Feature 3: Densité du cluster (nb de jours actifs comme proxy)
df_master['feat_densite_cluster'] = df_master['nb_jours_actifs'].fillna(0) / 90  # Normaliser sur 3 mois

# --- FEATURES DE L'AUTO-ENCODEUR ---

# Feature 4: Intensité de l'erreur (déjà normalisée)
df_master['feat_error_intensity'] = df_master['error_normalized']

# Feature 5: Durée de l'événement (calculée via groupement)
# Grouper les anomalies AE consécutives
df_master['ae_group'] = (df_master['anomaly_ae'] != df_master['anomaly_ae'].shift()).cumsum()
event_durations = df_master[df_master['anomaly_ae'] == 1].groupby('ae_group').size() * 15  # minutes
df_master['feat_event_duration'] = df_master['ae_group'].map(event_durations).fillna(0)

# Feature 6: Brutalité du gradient (vitesse de montée de l'erreur)
df_master['error_diff'] = df_master['reconstruction_error'].diff()
df_master['feat_gradient_brutality'] = df_master['error_diff'].abs().fillna(0)

# --- FEATURES CROISÉES ---

# Feature 7: Concordance (les deux modèles détectent simultanément?)
df_master['feat_concordance'] = np.where(
    (df_master['anomaly_detected'] == 1) & (df_master['anomaly_ae'] == 1),
    1, 0
)

# Feature 8: Heure de la journée
df_master['feat_hour_of_day'] = df_master['timestamp'].dt.hour

print("✅ 8 Features calculées :")
print("   [1] feat_persistance_cluster")
print("   [2] feat_proximite_heure_typique")
print("   [3] feat_densite_cluster")
print("   [4] feat_error_intensity")
print("   [5] feat_event_duration")
print("   [6] feat_gradient_brutality")
print("   [7] feat_concordance")
print("   [8] feat_hour_of_day")


ÉTAPE 2 : CALCUL DES 8 FEATURES DE FUSION
✅ 8 Features calculées :
   [1] feat_persistance_cluster
   [2] feat_proximite_heure_typique
   [3] feat_densite_cluster
   [4] feat_error_intensity
   [5] feat_event_duration
   [6] feat_gradient_brutality
   [7] feat_concordance
   [8] feat_hour_of_day


In [8]:
print("\n" + "="*70)
print("ÉTAPE 3 : FUSION PAR RÈGLES EXPERTES (TABLEAU DE DÉCISION)")
print("="*70)

def fusion_rules_expert(row):
    """
    Applique le tableau de décision exact du document.
    
    Retourne: (type_alerte, confiance)
    """
    stat = row['anomaly_detected']
    ae = row['anomaly_ae']
    persistance = row['feat_persistance_cluster']
    duree = row['feat_event_duration']
    
    # CAS 1: ✅ Stat ET ✅ AE
    if stat == 1 and ae == 1:
        if persistance > 0.70:
            return "Ombrage Fixe Confirmé", 95
        elif persistance >= 0.30:
            return "Ombrage Fixe Modéré", 85
        else:
            return "Ombrage Fixe Modéré", 85  # Pas explicite dans table, on garde 85
    
    # CAS 2: ✅ Stat MAIS ❌ AE
    if stat == 1 and ae == 0:
        if persistance > 0.70:
            return "Ombrage Fixe Léger", 75
        else:
            return "Faux Positif Potentiel", 40
    
    # CAS 3: ❌ Stat MAIS ✅ AE
    if stat == 0 and ae == 1:
        if duree < 120:  # < 2h
            return "Événement Ponctuel", 85
        else:
            return "Ombrage Émergent Non Capté", 60
    
    # CAS 4: ❌ Stat ET ❌ AE
    return "Normal", 0

# Application des règles
result = df_master.apply(fusion_rules_expert, axis=1, result_type='expand')
df_master['type_alerte'] = result[0]
df_master['confiance'] = result[1]

# Statistiques
print("\n📊 DISTRIBUTION DES ALERTES :")
print(df_master['type_alerte'].value_counts())



ÉTAPE 3 : FUSION PAR RÈGLES EXPERTES (TABLEAU DE DÉCISION)

📊 DISTRIBUTION DES ALERTES :
type_alerte
Normal                        221447
Événement Ponctuel             12120
Ombrage Émergent Non Capté     12021
Name: count, dtype: int64


In [9]:
print("\n" + "="*70)
print("ÉTAPE 4 : POST-TRAITEMENT ET AGRÉGATION")
print("="*70)

# A. Lissage temporel : Supprimer détections isolées < 5 minutes
df_master['alert_group'] = (df_master['type_alerte'] != df_master['type_alerte'].shift()).cumsum()
group_durations = df_master.groupby('alert_group').size() * 15  # minutes
df_master['group_duration'] = df_master['alert_group'].map(group_durations)

# Filtrer : garder seulement si durée >= 5 min OU confiance >= 80%
df_master['type_alerte_filtered'] = np.where(
    (df_master['group_duration'] >= 5) | (df_master['confiance'] >= 80),
    df_master['type_alerte'],
    'Normal'
)

print(f"✅ Lissage : {(df_master['type_alerte'] != df_master['type_alerte_filtered']).sum()} points supprimés (< 5min)")

# B. Validation de récurrence (pour Ombrage Fixe)
# Regrouper par jour et compter les jours avec alerte "Ombrage Fixe"
df_master['date'] = df_master['timestamp'].dt.date
ombrage_fixe_days = df_master[
    df_master['type_alerte_filtered'].str.contains('Ombrage Fixe', na=False)
].groupby('date').size()

total_days = df_master['date'].nunique()
recurrence_rate = len(ombrage_fixe_days) / total_days

print(f"✅ Récurrence Ombrage Fixe : {recurrence_rate*100:.1f}% des jours ({len(ombrage_fixe_days)}/{total_days} jours)")

# Si récurrence < 3/7 jours, reclasser
if recurrence_rate < 3/7:
    df_master['type_alerte_filtered'] = df_master['type_alerte_filtered'].replace(
        'Ombrage Fixe Confirmé', 'Événement Récurrent Faible'
    )
    print("⚠️ Récurrence faible : reclassification en 'Événement Récurrent Faible'")

# C. Agrégation en événements
print("\n📋 AGRÉGATION EN ÉVÉNEMENTS...")

# Grouper les alertes consécutives de même type
df_master['event_group'] = (
    (df_master['type_alerte_filtered'] != df_master['type_alerte_filtered'].shift()) |
    (df_master['timestamp'].diff() > pd.Timedelta('30min'))  # Break si gap > 30min
).cumsum()

# Créer la liste des événements
events = []
for group_id, group in df_master[df_master['type_alerte_filtered'] != 'Normal'].groupby('event_group'):
    event = {
        'event_id': f"EVT_{group['timestamp'].min().strftime('%Y_%m_%d_%H%M')}",
        'timestamp_debut': group['timestamp'].min(),
        'timestamp_fin': group['timestamp'].max(),
        'duree_minutes': (group['timestamp'].max() - group['timestamp'].min()).total_seconds() / 60,
        'type_principal': group['type_alerte_filtered'].mode()[0],
        'confiance_globale': group['confiance'].mean(),
        'source_detection': [],
        'perte_moyenne_pct': group['intensite_moyenne_pct'].mean() if 'intensite_moyenne_pct' in group else None,
        'nb_occurrences': len(group),
        'priorite': 'Haute' if group['confiance'].mean() > 85 else 'Moyenne'
    }
    
    # Déterminer sources
    if group['anomaly_detected'].sum() > 0:
        event['source_detection'].append('Statistique')
    if group['anomaly_ae'].sum() > 0:
        event['source_detection'].append('Auto-Encodeur')
    
    # Recommandations
    if 'Ombrage Fixe' in event['type_principal']:
        event['action_recommandee'] = 'Inspection terrain + Élagage'
    elif 'Événement Ponctuel' in event['type_principal']:
        event['action_recommandee'] = 'Surveillance continue'
    else:
        event['action_recommandee'] = 'Analyse approfondie'
    
    events.append(event)

df_events = pd.DataFrame(events)

print(f"✅ {len(df_events)} événements agrégés créés")



ÉTAPE 4 : POST-TRAITEMENT ET AGRÉGATION
✅ Lissage : 0 points supprimés (< 5min)
✅ Récurrence Ombrage Fixe : 0.0% des jours (0/2559 jours)
⚠️ Récurrence faible : reclassification en 'Événement Récurrent Faible'

📋 AGRÉGATION EN ÉVÉNEMENTS...
✅ 6176 événements agrégés créés


In [10]:
print("\n" + "="*70)
print("ÉTAPE 5 : EXPORT DES RÉSULTATS")
print("="*70)

# Sauvegarder DataFrame complet
df_master.to_csv('fusion_master_timeseries.csv', index=False)
print("💾 fusion_master_timeseries.csv")

# Sauvegarder événements agrégés
df_events.to_csv('fusion_events_final.csv', index=False)
print("💾 fusion_events_final.csv")

# Sauvegarder uniquement alertes haute priorité
df_high_priority = df_events[df_events['priorite'] == 'Haute'].sort_values('confiance_globale', ascending=False)
df_high_priority.to_csv('fusion_alerts_high_priority.csv', index=False)
print("💾 fusion_alerts_high_priority.csv")

# ==================== AFFICHAGE RÉSUMÉ ====================
print("\n" + "="*70)
print("📊 RÉSUMÉ FINAL DE LA FUSION")
print("="*70)

print(f"\n🔢 STATISTIQUES GLOBALES :")
print(f"  • Total points analysés : {len(df_master)}")
print(f"  • Période : {df_master['timestamp'].min()} → {df_master['timestamp'].max()}")
print(f"  • Événements détectés : {len(df_events)}")
print(f"  • Alertes haute priorité : {len(df_high_priority)}")

print(f"\n📈 CONTRIBUTION DES MODÈLES :")
stat_only = ((df_master['anomaly_detected'] == 1) & (df_master['anomaly_ae'] == 0)).sum()
ae_only = ((df_master['anomaly_detected'] == 0) & (df_master['anomaly_ae'] == 1)).sum()
both = df_master['feat_concordance'].sum()
print(f"  • Détections Stat seul : {stat_only}")
print(f"  • Détections AE seul : {ae_only}")
print(f"  • Concordance (les 2) : {both}")
print(f"  • Taux de concordance : {both/(stat_only+ae_only+both+1e-6)*100:.1f}%")

print(f"\n🎯 TOP 5 ÉVÉNEMENTS HAUTE CONFIANCE :")
print(df_high_priority[['event_id', 'type_principal', 'confiance_globale', 
                         'duree_minutes', 'action_recommandee']].head())

print("\n✅ FUSION TERMINÉE ! Fichiers générés :")
print("   1. fusion_master_timeseries.csv (série temporelle complète)")
print("   2. fusion_events_final.csv (événements agrégés)")
print("   3. fusion_alerts_high_priority.csv (alertes prioritaires)")


ÉTAPE 5 : EXPORT DES RÉSULTATS
💾 fusion_master_timeseries.csv
💾 fusion_events_final.csv
💾 fusion_alerts_high_priority.csv

📊 RÉSUMÉ FINAL DE LA FUSION

🔢 STATISTIQUES GLOBALES :
  • Total points analysés : 245588
  • Période : 2017-11-01 00:00:00 → 2024-11-02 04:45:00
  • Événements détectés : 6176
  • Alertes haute priorité : 0

📈 CONTRIBUTION DES MODÈLES :
  • Détections Stat seul : 0
  • Détections AE seul : 24141
  • Concordance (les 2) : 0
  • Taux de concordance : 0.0%

🎯 TOP 5 ÉVÉNEMENTS HAUTE CONFIANCE :
Empty DataFrame
Columns: [event_id, type_principal, confiance_globale, duree_minutes, action_recommandee]
Index: []

✅ FUSION TERMINÉE ! Fichiers générés :
   1. fusion_master_timeseries.csv (série temporelle complète)
   2. fusion_events_final.csv (événements agrégés)
   3. fusion_alerts_high_priority.csv (alertes prioritaires)
